# polygraphics-backend — free GPU demo on Google Colab

Spins up the full pipeline (SAM + MapAnything + Gaussian Splatting) on a free Colab T4 and exposes it on a public `trycloudflare.com` URL — **no accounts, no card, no tokens.**

**Always open this notebook from GitHub** (never an old copy you uploaded):  
[Open in Colab](https://colab.research.google.com/github/mohannadfarhoud/polygraphics-backend/blob/main/colab_demo.ipynb)

That link always loads `main` from GitHub. If you still see `git clone` in section 2, you’re on a stale notebook — use the link above.

## Before you click "Run all"
1. **Switch the runtime to GPU**: `Runtime > Change runtime type > T4 GPU`.
2. Then `Runtime > Run all`.

## What you get
- A public HTTPS URL that looks like `https://random-words-xyz.trycloudflare.com`
- Swagger at `<URL>/swagger`, health at `<URL>/health`
- MapAnything mesh + Gaussian Splatting at full GPU speed

## Free-tier limits to know
- ~12 h max session, ~90 min idle disconnect — keep this tab focused.
- All disk is wiped when the runtime stops. Download any `.glb` you want to keep.
- Re-running the **last cell** stops the previous server cleanly and reissues a fresh tunnel URL.

## 1. Verify GPU is attached

In [ ]:
!nvidia-smi || echo '\nNo GPU detected. Switch Runtime > Change runtime type > T4 GPU and try again.'

## 2. Get the repository

Downloads **`main` as a ZIP** from GitHub (public repo — no login). **We do not use `git clone` here** so Colab never hits the “could not read Username” error.

Re-running “Run all” skips the download if `/content/polygraphics-backend/scripts/colab_setup.sh` already exists.

In [ ]:
%%bash
set -eu
REPO=/content/polygraphics-backend
OWNER=mohannadfarhoud
NAME=polygraphics-backend
BRANCH=main
ZIP=/tmp/polygraphics-main.zip
# Use codeload directly (final URL behind the github.com redirect) to avoid
# wget redirect/TLS issues that surface as "exit 8" on some Colab images.
URLS=(
  "https://codeload.github.com/${OWNER}/${NAME}/zip/refs/heads/${BRANCH}"
  "https://github.com/${OWNER}/${NAME}/archive/refs/heads/${BRANCH}.zip"
  "https://api.github.com/repos/${OWNER}/${NAME}/zipball/${BRANCH}"
)

if [ -f "$REPO/scripts/colab_setup.sh" ]; then
  echo "Repo already present — skipping download."
else
  rm -rf "$REPO" "$ZIP"
  ok=0
  for u in "${URLS[@]}"; do
    echo "Trying $u"
    if curl -fL --retry 3 --retry-delay 2 -A "polygraphics-colab" -o "$ZIP" "$u"; then
      ok=1
      break
    fi
    echo "  -> failed, trying next URL"
  done
  if [ "$ok" -ne 1 ]; then
    echo "ERROR: all download URLs failed. GitHub may be rate-limiting this Colab IP."
    echo "Wait ~60s and re-run, or paste the manual fallback from the README."
    exit 1
  fi

  echo "Downloaded $(stat -c%s "$ZIP" 2>/dev/null || stat -f%z "$ZIP") bytes; extracting..."
  apt-get install -y -qq unzip >/dev/null 2>&1 || true
  unzip -q -o "$ZIP" -d /content/
  rm -f "$ZIP"
  # Match either "<name>-<branch>" (codeload/github.com) or "<owner>-<name>-<sha>" (api zipball).
  src="$(ls -d /content/${NAME}-* 2>/dev/null | head -n1)"
  if [ -z "$src" ]; then
    src="$(ls -d /content/${OWNER}-${NAME}-* 2>/dev/null | head -n1)"
  fi
  if [ -z "$src" ]; then
    echo "ERROR: extracted folder not found under /content/. Got:"
    ls -la /content/
    exit 1
  fi
  mv "$src" "$REPO"
fi

cd "$REPO"
test -f scripts/colab_setup.sh
echo "OK — repo at $PWD"

## 3. Install everything (MapAnything, SAM, deps, checkpoints)

Takes ~3–5 minutes the first time. Re-running is fast (skips what's already there).

In [ ]:
%%bash
cd /content/polygraphics-backend
bash scripts/colab_setup.sh

## 4. (Optional) tweak runtime settings

Defaults are sensible. Uncomment to e.g. force the Gaussian-Splatting backend or raise iterations.

In [ ]:
import json
from pathlib import Path

p = Path('/content/polygraphics-backend/config/runtime_settings.json')
settings = json.loads(p.read_text())

# settings['reconstruction_backend'] = 'gaussian_splatting'  # force .ply path
# settings['gs_iterations'] = 7000                            # quick GS run
# settings['max_image_side'] = 1024

p.write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))

## 5. Start the API + open the public tunnel

**This cell stays running.** Watch the output for a line like:
```
https://random-words-xyz.trycloudflare.com
```
That's your public demo URL. Open `<URL>/swagger` in any browser to test.

Stop the cell to tear down the tunnel; re-run it to get a fresh URL.

In [ ]:
!bash /content/polygraphics-backend/scripts/colab_serve.sh